# **Bonus Project:** Classifying NSL-KDD Dataset using DRL

Classify network traffic from the NSL-KDD dataset as normal (0) or anomalous (1) using Deep Reinforcement Learning (DRL).

A Deep Q-Network (DQN) will be used to train a reinforcement learning agent, which learns to classify traffic based on feedback from its actions.

The model's goal is to optimize its policy and accurately identify normal vs. attack traffic for intrusion detection.


## Exploring Phase

In [20]:
import numpy as np
import pandas as pd

In [21]:
df = pd.read_csv('KDD_Shuffled_Combined_Set.csv')

In [22]:
df

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


,duration,protocol_type,service,flag,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,...,dst_host_srv_count,dst_host_same_srv_rate,dst_host_diff_srv_rate,dst_host_same_src_port_rate,dst_host_srv_diff_host_rate,dst_host_serror_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,dst_host_srv_rerror_rate,label
0,0,tcp,http,SF,338,18918,0,0,0,0,...,255,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.0
1,0,tcp,http,SF,308,4336,0,0,0,0,...,255,1.00,0.00,0.02,0.03,0.00,0.00,0.00,0.00,0.0
2,0,tcp,smtp,SF,1064,338,0,0,0,0,...,198,0.80,0.04,0.01,0.01,0.00,0.00,0.00,0.00,0.0
3,0,tcp,smtp,SF,768,384,0,0,0,0,...,117,0.59,0.03,0.01,0.02,0.01,0.00,0.00,0.00,0.0
4,0,tcp,smtp,S0,0,0,0,0,0,0,...,118,0.39,0.05,0.02,0.02,0.02,0.02,0.01,0.01,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
148511,0,tcp,echo,RSTO,0,0,0,0,0,0,...,4,0.02,0.09,0.00,0.00,0.00,0.00,1.00,1.00,1.0
148512,0,tcp,telnet,S0,0,0,0,0,0,0,...,4,0.02,0.07,0.01,0.00,1.00,1.00,0.00,0.00,1.0
148513,0,tcp,smtp,SF,0,83,0,0,0,0,...,124,0.49,0.03,0.00,0.00,0.00,0.00,0.00,0.00,0.0
148514,899,tcp,domain,RSTO,1562,0,0,0,0,0,...,2,1.00,0.00,0.50,0.00,0.00,0.00,1.00,1.00,1.0


In [23]:
df.isnull().sum()

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


,0
duration,0
protocol_type,0
service,0
flag,0
src_bytes,0
dst_bytes,0
land,0
wrong_fragment,0
urgent,0
hot,0


In [24]:
df.dtypes

,0
duration,int64
protocol_type,object
service,object
flag,object
src_bytes,int64
dst_bytes,int64
land,int64
wrong_fragment,int64
urgent,int64
hot,int64


In [25]:
for col_name in ('label', 'protocol_type', 'service', 'flag'):
    unique = df[col_name].unique()
    print(col_name, '\t', len(unique), unique)
    print()

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


label 	 2 [0. 1.]

protocol_type 	 3 ['tcp' 'udp' 'icmp']

service 	 70 ['http' 'smtp' 'domain_u' 'other' 'courier' 'private' 'ftp_data' 'telnet'
 'eco_i' 'link' 'auth' 'bgp' 'sql_net' 'whois' 'imap4' 'tftp_u' 'gopher'
 'ecr_i' 'netbios_ssn' 'Z39_50' 'X11' 'http_443' 'time' 'ftp' 'uucp_path'
 'pop_3' 'klogin' 'efs' 'discard' 'ssh' 'netbios_dgm' 'supdup' 'uucp'
 'login' 'urp_i' 'kshell' 'echo' 'finger' 'hostnames' 'netstat' 'domain'
 'nnsp' 'ldap' 'iso_tsap' 'ctf' 'pop_2' 'nntp' 'exec' 'daytime' 'vmnet'
 'csnet_ns' 'netbios_ns' 'systat' 'mtp' 'ntp_u' 'remote_job' 'name' 'IRC'
 'rje' 'urh_i' 'sunrpc' 'printer' 'red_i' 'shell' 'tim_i' 'aol' 'pm_dump'
 'harvest' 'http_2784' 'http_8001']

flag 	 11 ['SF' 'S0' 'REJ' 'RSTR' 'RSTO' 'S1' 'SH' 'S2' 'S3' 'OTH' 'RSTOS0']



## Preprocessing Phase

In [26]:

col_names = ['protocol_type', 'service', 'flag']

df_encoded = pd.get_dummies(df, columns=col_names)
df_encoded

,duration,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,num_failed_logins,logged_in,num_compromised,...,flag_REJ,flag_RSTO,flag_RSTOS0,flag_RSTR,flag_S0,flag_S1,flag_S2,flag_S3,flag_SF,flag_SH
0,0,338,18918,0,0,0,0,0,1,0,...,False,False,False,False,False,False,False,False,True,False
1,0,308,4336,0,0,0,0,0,1,0,...,False,False,False,False,False,False,False,False,True,False
2,0,1064,338,0,0,0,0,0,1,0,...,False,False,False,False,False,False,False,False,True,False
3,0,768,384,0,0,0,0,0,1,0,...,False,False,False,False,False,False,False,False,True,False
4,0,0,0,0,0,0,0,0,0,0,...,False,False,False,False,True,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
148511,0,0,0,0,0,0,0,0,0,0,...,False,True,False,False,False,False,False,False,False,False
148512,0,0,0,0,0,0,0,0,0,0,...,False,False,False,False,True,False,False,False,False,False
148513,0,0,83,0,0,0,0,0,1,0,...,False,False,False,False,False,False,False,False,True,False
148514,899,1562,0,0,0,0,0,0,0,0,...,False,True,False,False,False,False,False,False,False,False


In [27]:
df_encoded.dtypes

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


,0
duration,int64
src_bytes,int64
dst_bytes,int64
land,int64
wrong_fragment,int64
...,...
flag_S1,bool
flag_S2,bool
flag_S3,bool
flag_SF,bool


In [28]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
df_scaled = pd.DataFrame(scaler.fit_transform(df_encoded), columns=df_encoded.columns)
df_scaled['label'] = df_encoded['label']
df_scaled['label']

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


,label
0,0.0
1,0.0
2,0.0
3,0.0
4,0.0
...,...
148511,1.0
148512,1.0
148513,0.0
148514,1.0


In [29]:
from sklearn.model_selection import train_test_split

X = df_scaled.drop('label', axis=1)
y = df_scaled['label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


## Creating Environment

In [30]:
import gym
from gym import spaces
import random

class NSLKDDEnv(gym.Env):
    def __init__(self, X, y):
        super(NSLKDDEnv, self).__init__()

        # Attached dataset can be test or train depending on the phase
        self.X = X.reset_index(drop=True)
        self.y = y.reset_index(drop=True)

        # Initial State
        self.reset()

        # Actions are Normal and Anomaly
        self.action_space = spaces.Discrete(2)

        # Space is simply the features
        self.observation_space = spaces.Box(low=-np.inf, high=np.inf, shape=(self.X.shape[1],), dtype=np.float32)

    def reset(self):
        # Randomly select any row from the linked dataset
        self.loc = 0
        self.perc = 0.0
        self.state = self.X.loc[self.loc]
        return self.state

    def step(self, action):
        # Postive reward if correct action
        # Negative reward if wrong action
        reward = 1 if action == self.y.loc[self.loc] else -1

        # Next location
        self.loc += 1
        self.perc = self.loc / self.X.shape[0]

        # Done if there is no more remaining locations
        done = True
        if self.loc < self.X.shape[0]:
            self.state = self.X.loc[self.loc]
            done = False

        return self.state, reward, done, {}

    def render(self):
        print(f'NSLKDDEnv at {self.perc:%} of steps')

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


In [31]:
env = NSLKDDEnv(X_train, y_train)

state = env.reset()
done = False
score = 0
while not done:
    action = env.action_space.sample()  # Random action (select feature index)
    new_state, reward, done, _ = env.step(action)
    score += reward
    print(f'Score: {score}', end='\t')
    env.render()
    state = new_state
    if env.loc == 10:
        break

Score: -1	NSLKDDEnv at 0.000842% of steps
Score: -2	NSLKDDEnv at 0.001683% of steps
Score: -3	NSLKDDEnv at 0.002525% of steps
Score: -2	NSLKDDEnv at 0.003367% of steps
Score: -3	NSLKDDEnv at 0.004208% of steps
Score: -2	NSLKDDEnv at 0.005050% of steps
Score: -1	NSLKDDEnv at 0.005892% of steps
Score: 0	NSLKDDEnv at 0.006733% of steps
Score: -1	NSLKDDEnv at 0.007575% of steps
Score: -2	NSLKDDEnv at 0.008417% of steps


## Training Phase, DRL Model

In [32]:
import tensorflow as tf
from tensorflow.keras import models, layers, optimizers

In [33]:
input_shape = env.observation_space.shape[0]
num_actions = env.action_space.n
input_shape, num_actions

(122, 2)

In [34]:
!pip install stable-baselines3 torch shimmy

In [35]:
from stable_baselines3 import PPO

env = NSLKDDEnv(X_train, y_train)
model = PPO('MlpPolicy', env, verbose=1)
model.learn(total_timesteps=1000000)

Using cuda device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.


/usr/local/lib/python3.10/dist-packages/stable_baselines3/common/vec_env/patch_gym.py:49: UserWarning: You provided an OpenAI Gym environment. We strongly recommend transitioning to Gymnasium environments. Stable-Baselines3 is automatically wrapping your environments in a compatibility layer, which could potentially cause issues.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/stable_baselines3/common/on_policy_algorithm.py:150: UserWarning: You are trying to run PPO on the GPU, but it is primarily intended to run on the CPU when not using a CNN policy (you are using ActorCriticPolicy which should be a MlpPolicy). See https://github.com/DLR-RM/stable-baselines3/issues/1245 for more info. You can pass `device='cpu'` or `export CUDA_VISIBLE_DEVICES=` to force using the CPU.Note: The model will train, but the GPU utilization will be poor and the training might take longer than on CPU.
  warnings.warn(


-----------------------------
| time/              |      |
|    fps             | 364  |
|    iterations      | 1    |
|    time_elapsed    | 5    |
|    total_timesteps | 2048 |
-----------------------------
-----------------------------------------
| time/                   |             |
|    fps                  | 303         |
|    iterations           | 2           |
|    time_elapsed         | 13          |
|    total_timesteps      | 4096        |
| train/                  |             |
|    approx_kl            | 0.018240504 |
|    clip_fraction        | 0.376       |
|    clip_range           | 0.2         |
|    entropy_loss         | -0.678      |
|    explained_variance   | -0.0274     |
|    learning_rate        | 0.0003      |
|    loss                 | 3.4         |
|    n_updates            | 10          |
|    policy_gradient_loss | -0.0483     |
|    value_loss           | 7.21        |
-----------------------------------------
----------------------------------

KeyboardInterrupt: 

In [36]:
model.save("drl_model")

In [41]:
from sklearn.metrics import accuracy_score, balanced_accuracy_score, precision_score, recall_score, f1_score, matthews_corrcoef

model = PPO.load("drl_model")

# Evaluate the agent
envs = {'train': NSLKDDEnv(X_train, y_train), 'test': NSLKDDEnv(X_test, y_test)}
for phase in ['train', 'test']:
    print(f"\n\nEvaluating {phase} phase...")

    env = envs[phase]
    obs = env.reset()
    done = False
    total_rewards = 0
    predicted_labels = []
    true_labels = env.y
    pred = []

    while not done:
        action, _state = model.predict(obs)
        obs, reward, done, info = env.step(action)
        total_rewards += reward
        predicted_labels.append(action)
        perc = env.perc * 100
        if perc > 0 and perc % 5 == 0:
          env.render()

    print(f"Total rewards for {phase} phase: {total_rewards}")

    accuracy = accuracy_score(true_labels, predicted_labels)
    balanced_accuracy = balanced_accuracy_score(true_labels, predicted_labels)
    precision = precision_score(true_labels, predicted_labels, zero_division=1)
    recall = recall_score(true_labels, predicted_labels, zero_division=1)
    f1 = f1_score(true_labels, predicted_labels, zero_division=1)
    mcc = matthews_corrcoef(true_labels, predicted_labels)

    # Print metrics
    print("Evaluation Metrics:")
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Balanced Accuracy: {balanced_accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1-Score: {f1:.4f}")
    print(f"Matthews Correlation Coefficient (MCC): {mcc:.4f}")


    env.close()

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)
/usr/local/lib/python3.10/dist-packages/stable_baselines3/common/on_policy_algorithm.py:150: UserWarning: You are trying to run PPO on the GPU, but it is primarily intended to run on the CPU when not using a CNN policy (you are using ActorCriticPolicy which should be a MlpPolicy). See https://github.com/DLR-RM/stable-baselines3/issues/1245 for more info. You can pass `device='cpu'` or `export CUDA_VISIBLE_DEVICES=` to force using the CPU.Note: The model will train, but the GPU utilization will be poor and the training might take longer than on CPU.
  warnings.warn(




Evaluating train phase...
NSLKDDEnv at 25.000000% of steps
NSLKDDEnv at 50.000000% of steps
NSLKDDEnv at 75.000000% of steps
NSLKDDEnv at 100.000000% of steps
Total rewards for train phase: 111794
Evaluation Metrics:
Accuracy: 0.9705
Balanced Accuracy: 0.9702
Precision: 0.9757
Recall: 0.9625
F1-Score: 0.9691
Matthews Correlation Coefficient (MCC): 0.9409


Evaluating test phase...
NSLKDDEnv at 25.000000% of steps
NSLKDDEnv at 50.000000% of steps
NSLKDDEnv at 75.000000% of steps
NSLKDDEnv at 100.000000% of steps
Total rewards for test phase: 27912
Evaluation Metrics:
Accuracy: 0.9698
Balanced Accuracy: 0.9696
Precision: 0.9753
Recall: 0.9620
F1-Score: 0.9686
Matthews Correlation Coefficient (MCC): 0.9397


### Train Phase
| Metric                           | Value       |
|----------------------------------|-------------|
| Total Rewards                    | 111794      |
| Accuracy                         | 0.9705      |
| Balanced Accuracy                | 0.9702      |
| Precision                        | 0.9757      |
| Recall                           | 0.9625      |
| F1-Score                         | 0.9691      |
| Matthews Correlation Coefficient (MCC) | 0.9409  |

### Test Phase
| Metric                           | Value       |
|----------------------------------|-------------|
| Total Rewards                    | 27912       |
| Accuracy                         | 0.9698      |
| Balanced Accuracy                | 0.9696      |
| Precision                        | 0.9753      |
| Recall                           | 0.9620      |
| F1-Score                         | 0.9686      |
| Matthews Correlation Coefficient (MCC) | 0.9397  |